# Capacity feature ablation — unconstrained BiLSTM gap-imputer

## Setup

In [1]:
import os, math, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

TARGETS = ["hydro", "coal_brown", "gas_steam", "gas_ocgt",
           "battery_charging", "battery_discharging",
           "wind", "solar_utility", "curtailment_wind", "curtailment_solar"]
OBS = ["demand", "price"]                       # known everywhere (not gapped)
CAL = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]
GEN_FT = ["hydro", "coal_brown", "gas_steam", "gas_ocgt",
          "battery_charging", "battery_discharging", "wind", "solar_utility"]

STEPS_PER_DAY = 288
GAP = 36                                         # 3h gap (36 x 5min)
GAP_EVAL_START = 132                             # eval gap pinned to 11:00-14:00
SPLIT = (0.8, 0.1, 0.1)                          # chronological train/val/test
SEED = 0
HIDDEN, LAYERS, DROPOUT = 128, 2, 0.2
LR, WEIGHT_DECAY, BATCH = 1e-3, 1e-4, 64
EPOCHS, PATIENCE = 100, 12

def seed_all(s=SEED):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all()

_mps = getattr(torch.backends, "mps", None)
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if (_mps is not None and _mps.is_available()) else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, "| cuda:", torch.cuda.is_available(),
      "(" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "")

ModuleNotFoundError: No module named 'pandas'

## Data

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    # upload the 4 updata parquets to /content (or a /content/updata folder)
    DATA_DIR = Path("/content/updata") if Path("/content/updata").exists() else Path("/content")
else:
    DATA_DIR = Path.cwd()
    while DATA_DIR != DATA_DIR.parent and not (DATA_DIR / "data" / "updata").exists():
        DATA_DIR = DATA_DIR.parent
    DATA_DIR = DATA_DIR / "data" / "updata"

F_GEN = DATA_DIR / "vic_generation_20250101_20260814.parquet"
F_MKT = DATA_DIR / "vic_market_20250101_20260814.parquet"
F_CUR = DATA_DIR / "vic_curtailment_20250101_20260814.parquet"
F_CAP = DATA_DIR / "capacity_VIC1_by_fueltech_MS.parquet"
print("IN_COLAB:", IN_COLAB, "| DATA_DIR:", DATA_DIR)
for f in (F_GEN, F_MKT, F_CUR, F_CAP):
    print("  ", f.name, "OK" if f.exists() else "MISSING")

def _naive(idx):
    idx = pd.DatetimeIndex(idx)
    return idx.tz_localize(None) if idx.tz is not None else idx

g = pd.read_parquet(F_GEN); g["interval"] = _naive(g["interval"])
gw = (g.pivot_table(index="interval", columns="fueltech", values="power_mw", aggfunc="first")
        .reindex(columns=GEN_FT))
m = pd.read_parquet(F_MKT); m["interval"] = _naive(m["interval"])
m = m.set_index("interval").rename(columns={"demand_mw": "demand", "price_aud_per_mwh": "price"})[OBS]
c = pd.read_parquet(F_CUR); c["interval"] = _naive(c["interval"])
c = (c.set_index("interval")
       .rename(columns={"curtailment_wind_mw": "curtailment_wind",
                        "curtailment_solar_mw": "curtailment_solar"})
       [["curtailment_wind", "curtailment_solar"]])
df = gw.join(m, how="outer").join(c, how="outer").sort_index()

h = df.index.hour + df.index.minute / 60.0
df["hour_sin"], df["hour_cos"] = np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)
dow = df.index.dayofweek
df["dow_sin"], df["dow_cos"] = np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)

cap = pd.read_parquet(F_CAP); cap["interval"] = _naive(cap["interval"])
capw = (cap.pivot_table(index="interval", columns="fueltech", values="capacity_mw", aggfunc="first")
          .reindex(df.index, method="ffill"))
capw.columns = [f"cap_{col}" for col in capw.columns]
df = df.join(capw)
print("table:", df.shape, "|", df.index.min(), "->", df.index.max())
df.head()

## Features

In [ ]:
CAP_COLS = [c for c in df.columns if c.startswith("cap_")]
cols_A = TARGETS + OBS + CAL                     # arm A: no capacity
cols_B = TARGETS + OBS + CAL + CAP_COLS          # arm B: + per-fueltech capacity

def make_days(frame, cols):
    """Whole-day tensors (D, 288, C) over complete, clean days only."""
    sub = frame[cols].copy()
    for cc in ("curtailment_wind", "curtailment_solar"):
        if cc in sub: sub[cc] = sub[cc].fillna(0.0)
    sub = sub.interpolate(limit=6, limit_direction="both")
    day = sub.index.normalize()
    arrs, days = [], []
    for d, g_idx in sub.groupby(day).groups.items():
        block = sub.loc[g_idx].sort_index()
        if len(block) != STEPS_PER_DAY or block.isna().any().any():
            continue
        arrs.append(block.to_numpy(np.float32)); days.append(pd.Timestamp(d))
    return np.stack(arrs), pd.DatetimeIndex(days)

dfA, dfB = df[cols_A], df[cols_B]                # the two feature frames
XA, daysA = make_days(df, cols_A)
XB, daysB = make_days(df, cols_B)
assert daysA.equals(daysB), "arm A/B day sets differ -> windows would not match"
print("arm A:", XA.shape, "| arm B:", XB.shape, "| clean days:", len(daysA))
dfB.head()

## Split

In [ ]:
D = len(daysA)
n_tr, n_va = int(D * SPLIT[0]), int(D * SPLIT[1])
tr, va, te = slice(0, n_tr), slice(n_tr, n_tr + n_va), slice(n_tr + n_va, D)
print(f"chronological split  train={n_tr}  val={n_va}  test={D - n_tr - n_va}")
print(f"  train {daysA[tr][0].date()} .. {daysA[tr][-1].date()}")
print(f"  val   {daysA[va][0].date()} .. {daysA[va][-1].date()}")
print(f"  test  {daysA[te][0].date()} .. {daysA[te][-1].date()}")

## Scaling

In [ ]:
def scale_arm(X, cols):
    """Fit per-channel mean/std on TRAIN only; drop constant (zero-variance) scaled cols."""
    ci = {c: i for i, c in enumerate(cols)}
    scale_cols = TARGETS + OBS + [c for c in cols if c.startswith("cap_")]
    scale_idx = [ci[c] for c in scale_cols]
    tgt_idx = [ci[c] for c in TARGETS]
    flat = X[tr].reshape(-1, X.shape[2]).astype(np.float64)     # f64: exact const detect
    mean, std = flat.mean(0), flat.std(0)
    cmin, cmax = flat.min(0), flat.max(0)
    sset = set(scale_idx)
    const = (cmax - cmin) <= 1e-9 * (np.abs(cmax) + 1.0)        # range-based, f32-safe
    keep = np.array([not (j in sset and const[j]) for j in range(len(cols))])
    dropped = [cols[j] for j in range(len(cols)) if not keep[j]]
    Xs = X.astype(np.float32).copy()
    for j in scale_idx:
        if keep[j]:
            Xs[:, :, j] = (X[:, :, j] - mean[j]) / std[j]
    Xs = Xs[:, :, keep]
    cols2 = [c for j, c in enumerate(cols) if keep[j]]
    return {"Xs": Xs, "cols": cols2, "tgt_idx": [cols2.index(c) for c in TARGETS],
            "y_mean": mean[tgt_idx].astype(np.float32),
            "y_std": std[tgt_idx].astype(np.float32), "dropped": dropped}

arms = {"no_cap": scale_arm(XA, cols_A), "cap": scale_arm(XB, cols_B)}
for k, a in arms.items():
    print(f"{k}: channels={len(a['cols'])}  dropped_const={a['dropped']}")

## Windows

In [ ]:
def masked_batch(Xs, tgt_idx, starts):
    """Zero the target channels inside each 3h gap and build the mask channel."""
    B, T, C = Xs.shape
    xin = Xs.copy(); mask = np.ones((B, T, 1), np.float32)
    gidx = np.stack([np.arange(int(s), int(s) + GAP) for s in starts])
    for b in range(B):
        s = int(starts[b])
        for t in tgt_idx:
            xin[b, s:s + GAP, t] = 0.0
        mask[b, s:s + GAP, 0] = 0.0
    return xin.astype(np.float32), mask, gidx

def rand_starts(B, rng):
    return rng.integers(0, STEPS_PER_DAY - GAP, size=B)

def fixed_starts(B):
    return np.full(B, GAP_EVAL_START, dtype=int)

## Model

In [ ]:
class BiLSTMImputer(nn.Module):
    """Unconstrained bidirectional-LSTM gap imputer: features(+mask) -> target channels."""
    def __init__(self, n_in, n_tgt, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT):
        super().__init__()
        self.rnn = nn.LSTM(n_in + 1, hidden, layers, batch_first=True,
                           bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden * 2, n_tgt)

    def forward(self, x, mask):
        h, _ = self.rnn(torch.cat([x, mask], dim=-1))
        return self.head(h)                      # (B, T, n_tgt) in TARGETS order

## Training

In [ ]:
def train_arm(arm, name):
    seed_all()
    Xtr, Xva = arm["Xs"][tr], arm["Xs"][va]
    tgt = arm["tgt_idx"]; tsel = np.asarray(tgt)
    model = BiLSTMImputer(Xtr.shape[2], len(tgt)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    vin, vmask, vg = masked_batch(Xva, tgt, fixed_starts(len(Xva)))   # fixed val gaps
    vin_t = torch.from_numpy(vin).to(DEVICE); vmask_t = torch.from_numpy(vmask).to(DEVICE)
    vy = torch.from_numpy(Xva[..., tsel]).to(DEVICE)
    vgsel = torch.zeros(len(Xva), STEPS_PER_DAY, dtype=torch.bool)
    for b in range(len(Xva)): vgsel[b, vg[b]] = True
    vgsel = vgsel.to(DEVICE)

    hist = {"train": [], "val": []}; best, best_state, waited = 1e9, None, 0
    for ep in range(1, EPOCHS + 1):
        rng = np.random.default_rng(1000 + ep)                       # fresh gaps / epoch
        order = rng.permutation(len(Xtr))
        model.train(); tl = []
        for i in range(0, len(Xtr), BATCH):
            bidx = order[i:i + BATCH]; xb0 = Xtr[bidx]
            xin, mask, gid = masked_batch(xb0, tgt, rand_starts(len(bidx), rng))
            xb = torch.from_numpy(xin).to(DEVICE); mb = torch.from_numpy(mask).to(DEVICE)
            yb = torch.from_numpy(xb0[..., tsel]).to(DEVICE)
            gsel = torch.zeros(len(bidx), STEPS_PER_DAY, dtype=torch.bool)
            for j in range(len(bidx)): gsel[j, gid[j]] = True
            gsel = gsel.to(DEVICE)
            opt.zero_grad()
            loss = ((model(xb, mb) - yb)[gsel] ** 2).mean()
            loss.backward(); opt.step(); tl.append(loss.item())
        model.eval()
        with torch.no_grad():
            vloss = ((model(vin_t, vmask_t) - vy)[vgsel] ** 2).mean().item()
        hist["train"].append(float(np.mean(tl))); hist["val"].append(vloss)
        if vloss < best - 1e-6:
            best, waited = vloss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
        if ep % 10 == 0 or waited == 0:
            print(f"  [{name}] ep{ep:03d} train={hist['train'][-1]:.4f} val={vloss:.4f} best={best:.4f}")
        if waited >= PATIENCE:
            print(f"  [{name}] early stop ep{ep} best_val={best:.4f}"); break
    if best_state is not None: model.load_state_dict(best_state)
    return model, hist

models = {}
for name, arm in arms.items():
    print(f"train arm: {name}  (in={arm['Xs'].shape[2]} channels)")
    models[name], arm["hist"] = train_arm(arm, name)

## Evaluation

In [ ]:
def metrics(P, Y, targets):
    """Per-channel WAPE/MAE/RMSE/R2 + macro (nanmean); P,Y are (N, GAP, nt) in MW."""
    p, y = P.reshape(-1, len(targets)), Y.reshape(-1, len(targets))
    out = {}
    for i, t in enumerate(targets):
        e = p[:, i] - y[:, i]; den = np.abs(y[:, i]).sum()
        sstot = ((y[:, i] - y[:, i].mean()) ** 2).sum()
        out[t] = {"WAPE": float(np.abs(e).sum() / den) if den > 0 else np.nan,
                  "MAE": float(np.abs(e).mean()),
                  "RMSE": float(np.sqrt((e ** 2).mean())),
                  "R2": float(1 - (e ** 2).sum() / sstot) if sstot > 0 else np.nan}
    macro = {mk: float(np.nanmean([out[t][mk] for t in targets]))
             for mk in ["WAPE", "MAE", "RMSE", "R2"]}
    return out, macro

def eval_arm(model, arm, day_subset=None):
    Xte = arm["Xs"][te]; tgt = arm["tgt_idx"]
    if day_subset is not None: Xte = Xte[day_subset]
    xin, mask, gid = masked_batch(Xte, tgt, fixed_starts(len(Xte)))
    model.eval()
    with torch.no_grad():
        pred = model(torch.from_numpy(xin).to(DEVICE),
                     torch.from_numpy(mask).to(DEVICE)).cpu().numpy()
    P = np.stack([pred[b, gid[b]] for b in range(len(Xte))]) * arm["y_std"] + arm["y_mean"]
    Y = np.stack([Xte[b, gid[b]][:, tgt] for b in range(len(Xte))]) * arm["y_std"] + arm["y_mean"]
    return metrics(P, Y, TARGETS)

# top-20 highest-demand test days (rank by within-day peak demand)
di = arms["no_cap"]["cols"].index("demand")
peak = arms["no_cap"]["Xs"][te][:, :, di].max(1)
top20 = np.argsort(-peak)[:20]

rows = []
for name, arm in arms.items():
    _, m_all = eval_arm(models[name], arm)
    _, m_top = eval_arm(models[name], arm, day_subset=top20)
    rows.append((name, m_all["WAPE"], m_all["R2"], m_top["WAPE"], m_top["R2"]))
res = pd.DataFrame(rows, columns=["arm", "test_WAPE", "test_R2", "top20_WAPE", "top20_R2"]).set_index("arm")
print(res.round(4).to_string())
print("\ndelta (cap - no_cap):  test_WAPE",
      round(res.loc["cap", "test_WAPE"] - res.loc["no_cap", "test_WAPE"], 4),
      "| top20_WAPE", round(res.loc["cap", "top20_WAPE"] - res.loc["no_cap", "top20_WAPE"], 4))